# Pre-processing a .CHA file for use in CE analysis

In [ ]:
import os
import sys
import pandas as pd
from tqdm import tqdm

In [ ]:
data_path = '../data'
chas_path = os.path.join(data_path, 'chas')
outputs_path = os.path.join(data_path, 'server_ready', 'cha_corpus.csv')

In [ ]:
def grab_all_files(PATH, file_type='.cha'):
    files = [
        [
            os.path.join(root, f) for f in files 
            if f.endswith(file_type) and (not f.startswith('._'))
        ] 
        for root, _, files in os.walk(PATH) 
    ]
    return sum(files, [])

# Processing all CHA files

Note: the package used here was developed by Prof. Garber. Cite via:

Garber, L. (2019). CHA file python parser. Zenodo. https://doi.org/10.5281/zenodo.3364020

In [ ]:
from shared.CHAFile import *

In [ ]:
all_files = grab_all_files(chas_path)
# all_files

In [ ]:
data = []
for f in all_files:
    chacha = ChaFile(f)
    meta_data_pieces = f.replace('.cha', '').split('/')
    for line in chacha.getLines():
        line['conversation_id'] = meta_data_pieces[-1]
        line['overlapping_text'] = bool(re.findall(r"(⌋|⌊|⌉|⌈)", line['text']))

        if meta_data_pieces[-2] in ['eng_n', 'eng_s']:
            line['corpus'] = meta_data_pieces[-3] + '-' + meta_data_pieces[-2]
        
        elif meta_data_pieces[-3] in ['instruction']:
            line['corpus'] = meta_data_pieces[-3] + '-' + meta_data_pieces[-2]
        
        else:
            line['corpus'] = meta_data_pieces[-2]
        
        data += [line.copy()]

data = pd.DataFrame(data)

In [ ]:
data.head()

### Correcting utterances/removing CLAN specific formatting.

In [ ]:
def corrected_text(text, contraction_replacement_nonce='CCOONNTTRRAACCTTIIOONN'):
    res = re.sub(r'\(\(.*?\)\)', ' ', str(text))
    # res = re.sub(r'\[.*?\]', ' ', res)
    
    # find contractions and preserve them . . .
    contractions = list(re.findall(r"\w+'\w+", res))
    for contraction in set(contractions):
        replacement = contraction.replace("'", contraction_replacement_nonce)
        res = res.replace(contraction, replacement)
    res = re.sub(r"(⌋|⌊|⌉|⌈)", '', res)
    res = res.replace(':', '')
    
    # remove numbers in parentheses (times???)
    res = re.sub(r'\(\d\.\d\)', ' ', res)
    
    # remove all other special characters.
    res = re.sub(r'[^\w\s]', ' ', res)
    
    res = re.sub(r'\s+', ' ', res).replace('[', ' ').replace(']', ' ').replace(contraction_replacement_nonce, "'")
    return res.strip()

In [ ]:
data['raw_text'] = data['text'].values
data['text'] = [corrected_text(text) for text in tqdm(data['raw_text'].values)]

In [ ]:
data[['corpus', 'raw_text', 'text']].head(n=6)

## Create juxtaposed corpus: (x,y) pairs

In [ ]:
max_turns_apart = 30

In [ ]:
import warnings; warnings.filterwarnings("ignore")

corpus = []
for cid in tqdm(data['conversation_id'].unique()):
    sub = data.loc[data['conversation_id'].isin([cid])]
    sub_index = sub.index.values
    
    for i in sub_index:
        if i != sub_index[-1]:
            
            # speaker vs. other
            next_line_no = ( (sub_index > i) & (~sub['speaker'].isin([sub['speaker'].loc[i]])) ).values.nonzero()[0]
            next_line_no = sub_index[next_line_no][:(max_turns_apart+1)]
            
            # next_line_no = next_line_no[next_line_no <= (i + max_turns_apart)]
            for j,li in enumerate(next_line_no):
                d = data.loc[i].to_dict()
                
                d['next_speaker'] = data['speaker'].loc[li]
                d['next_text'] = data['text'].loc[li]
                d['next_utterance_no'] = data['utterance_no'].loc[li]
                d['next_utterance_delta_no'] = j
                
                corpus += [d]
            
            # speaker vs. self 
            next_line_no = ( (sub_index > i) & (sub['speaker'].isin([sub['speaker'].loc[i]])) ).values.nonzero()[0]
            next_line_no = sub_index[next_line_no][:(max_turns_apart+1)]
            # next_line_no = next_line_no[next_line_no <= (i + max_turns_apart)]
            for j,li in enumerate(next_line_no):
                d = data.loc[i].to_dict()
                
                d['next_speaker'] = data['speaker'].loc[li]
                d['next_text'] = data['text'].loc[li]
                d['next_utterance_no'] = data['utterance_no'].loc[li]
                d['next_utterance_delta_no'] = j
                
                corpus += [d]

In [ ]:
data = pd.DataFrame(corpus)
data.head()

In [ ]:
rename_columns = dict()
for col in data.columns:
    if col == 'text':
        rename_columns[col] = 'x_utterance'
    elif col == 'next_text':
        rename_columns[col] = 'y_utterance'
    elif 'next_' in col:
        rename_columns[col]  = col.replace('next', 'y')
    else:
        rename_columns[col] = col

In [ ]:
data = data.rename(columns=rename_columns)
data.head()

In [ ]:
data = data.rename(columns={'speaker':'x_speaker','utterance_no': 'x_turn_id', 'y_utterance_no': 'y_turn_id'})
data.head()

In [ ]:
data['x_speaker'] = data['corpus'].astype(str) + '-' + data['conversation_id'].astype(str) + '-' + data['x_speaker'].astype(str)

data['y_speaker'] = data['corpus'].astype(str) + '-' + data['conversation_id'].astype(str) + '-' + data['y_speaker'].astype(str)

In [ ]:
data['self'] = data['x_speaker'] == data['y_speaker']
data = data.sort_values(by=['corpus', 'conversation_id', 'x_turn_id', 'self', 'y_turn_id'])
data.index = range(len(data))

In [ ]:
data[['corpus', 'x_utterance', 'y_utterance']].isna().mean()

In [ ]:
# corpus sanity check . . . make sure that conversation_ids are all unique 
k = data[['conversation_id', 'corpus']].drop_duplicates()
k['conversation_id'].nunique(), k['conversation_id'].nunique()/len(k)

In [ ]:
data[['corpus', 'x_utterance', 'x_speaker', 'y_speaker', 'y_utterance', 'x_turn_id', 'y_turn_id']].head()

In [ ]:
data.shape

## Save outputs for server operations.

In [ ]:
data.to_csv(outputs_path, index=False, encoding='utf-8')